# Unsupervised ML: Association Rule Mining & PCA
---

In this notebook we have a closer look at Assiciation Rule Mining and Principal Component Analysis, both are *unsupervised learning* techniques

### Overview
---

- [Association Rule Mining](#Association-Rule-Mining)
  - [Retrieving Transactional Data](#Retrieving-Transactional-Data)
  - [First Analysis](#First-Analysis)
  - [Mining the Rules](#Mining-the-Rules)
- [Principal Component Analysis (PCA)](#Principal-Component-Analysis-(PCA))

# Association Rule Mining

Let's have a close look on how to mine association rules in Python. Unfortunately, `scikit-learn` has no algorithms for that task available. However, the functionality was implemented in the library [Mlxtend](https://github.com/rasbt/mlxtend) by Sebastian Raschka. You can try to install this library using Anaconda now directly (since 2025):
> `conda install mlxtend`

Previously the library was not maintained by Anaconda, but the community. It requires to be installed from a different channel *conda-forge* using (try that if above does not work):
> `conda install -c conda-forge mlxtend`

A tutorial on how to use the library can also be found on this [website](http://pbpython.com/market-basket-analysis.html).

## Retrieving Transactional Data
---

As an example, we use the transactional data ("*Groceries*") from the R library [*arules*](https://cran.r-project.org/web/packages/arules/index.html), which has been converted to a CSV file. As the name implies, it contains data from **grocery shoppings**.

As always, let's first import the required Python libraries:

In [ ]:
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from mlxtend.preprocessing import TransactionEncoder # OnehotTransactions is *deprecated*
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

First, the data from the CSV file is read. Please note the different length of the transactions:

In [ ]:
with open('data/groceries_r_quoted.csv', 'r') as f:
    reader = csv.reader(f)
    transactions = list(reader)
transactions[:9] # print first 10 transactions

Now it is time to apply a [One Hot Encoding](http://rasbt.github.io/mlxtend/user_guide/preprocessing/TransactionEncoder/), such that every product is listed as an own column (dimension) containing a True/False value. Every row contains one transaction, but now all rows have the same length. We store the data in a new dataframe. The TransactionEncoder uses the known *fit-transform* method, similar to `scikit-learn`.

In [ ]:
oht = TransactionEncoder()
oht_ary = oht.fit(transactions).transform(transactions)
transactdf = pd.DataFrame(oht_ary, columns=oht.columns_) # use products as column names
transactdf.head()

**Remark:** `scikit-learn` also ships with a one-hot encoder [sklearn.preprocessing.OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), which output is 0/1 instead on True/False.

Statistics about amount of transactions and products:

In [ ]:
print("transactions:",len(transactdf))
print("products:",len(transactdf.columns))

Statistics about the size of transactions:

In [ ]:
transactdf.sum(axis=1).describe()

On average, a transaction has 4.4 items, but there are transactions of one item only and a maximum of 32. Since the median (50%) is at 3 items, there are probably many transactions with one or two items only. Let's check that:

In [ ]:
transactdf.sum(axis=1).value_counts().head()

Indeed, 2159 transactions contain one item only. They won't help us because there is no association to be mined. Finally, let's analyze the matrix and its items in a bit more detail in the next section.

## First Analysis
---

Obviously the matrix contains a lot of `False`-values due to the fact that only very few items are contained in single transactions. Such matrices, containing many NULL values (in our case False), are also called a **sparse matrix**. Let's try to measure how sparse our matrix is:

In [ ]:
nentries= transactdf.size
print("Total entries:",nentries)

In [ ]:
nnz = np.count_nonzero(transactdf)
print("Non-zero entries:",nnz)

In [ ]:
density = 100.0 * nnz / nentries
sparsity = 100-density # sparcity is amount_zero/entries
print("The matrix has ~{0:.2f}% non-NULL entries and thus a sparsity of ~{1:.2f}%.".format(density,sparsity))

For mining the association rules, the *support* (relative frequency) is computed first. For every item (column), sum up how often it was part of the transaction and devide by the total numer of transactions (element-wise division). Sort and print the result:

In [ ]:
supplist = (transactdf.sum()/len(transactdf)).sort_values(ascending=False)
supplist.head(10)

Filter items to have a minimum support of 10% (0.1)

In [ ]:
print(len(supplist[supplist>=0.1]))
supplist[supplist>=0.1]

In [ ]:
supplist[supplist>=0.1].plot(kind='bar', rot=90)
plt.show()

Maybe 10% was a bit harsh, let's plot the top-20 products according to their support.

In [ ]:
supplist.head(20).plot(kind='bar', rot=90, figsize=(10,5))
plt.show()

## Mining the Rules
---

To compute the *frequent item sets*, the [a-priori algorithm of Mlxtend](http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/) is used. The default value for the minimum support is 50% (0.5), which would lead to no results in our case. So let's set it to 0.004 (0.4%). Try to set it to some smaller value and observe the runtime!

In [ ]:
frequent_itemsets = apriori(transactdf, min_support=0.004, use_colnames=True, verbose=1)

In [ ]:
print (len(frequent_itemsets))
type(frequent_itemsets)

Show some entries from (the middle of) the dataframe:

In [ ]:
# frequent_itemsets[400:410] <- not so nice output including frozenset datatype,
# -> replace frozenset with set (just for noce output):
tmp = frequent_itemsets.copy()
tmp['itemsets'] = tmp['itemsets'].apply(set)
tmp[400:410]

Finally, the rules are generated. Since the head and body of the rule can be sets, we should again set a minimum value. To this end, a metric is selected (preferably *confidence* or *lift*) and a threshold is defined for it. More information about metrics can be found in the [docs](http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/).

In [ ]:
# just a helper function to replace the "frozenset" with a regular set for well-formatted output
def formatRulesToSets(r):
    tmp = r.copy()
    tmp['antecedents'] = tmp['antecedents'].apply(set)
    tmp['consequents'] = tmp['consequents'].apply(set)
    return tmp
    
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.25)
# rules = association_rules(frequent_itemsets, num_itemsets=len(frequent_itemsets), metric="confidence", min_threshold=0.25) <- for Bug in version 0.23.2
print (len(rules))
# print(rules.head()
formatRulesToSets(rules.head())


In total we got 998 rules.

Interestingly two new metrics show up here, *leverage* and *conviction*, which we did not talk about in the lecture (You can find out more in the [docs](http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/)). Please also note the different terms, the *head* of a rule is called *antecedent* and the body of a rule *consequent*.

Maybe we are interested in the absolute count of a decision rule in our dataset. We can do that easily by adding a new colum `count`, which contains the absolute number, the multiplication of support and the amount of transactions:

In [ ]:
rules["count"] = (rules['support']*len(transactdf)).astype(int)
formatRulesToSets(rules.head())

Please note that the support is the joint probability of both, so our absolute count contains not only the `antecedents->consequents` rules, but includes also `consequents->antecedents`.

Until now, the `head()` of the dataframe showed us an arbitrary subset of the rules. As we know from the lecture, it makes most sense to investigate the top rules according to the *lift*:

In [ ]:
formatRulesToSets(rules.sort_values("lift", ascending=False)[0:9])

It seems that the best rule (according to the *lift*) is: `flour -> sugar`. Please note that there are rules with more than one item in the head/body.

Inspired by the rule with the highest *lift* flour -> sugar, let's check whether there are more rules with flour in the head. Maybe people are baking cake!
Unfortunately, this task is not so easy to code since the value of antecedents is a set (in particular a `frozenset`, which is an immutable set):

In [ ]:
type(rules.antecedents.values[0])

In [ ]:
antecedents = rules.antecedents.values
contains_flour = ['flour' in fs for fs in antecedents] # True/False array if 'flour' is part of head of rule
formatRulesToSets(rules[contains_flour])

Seems like our "cake theory" is not correct...

Finally, let's check whether there are rules with a lift<1 (we know they are not good..):

In [ ]:
formatRulesToSets(rules[rules.lift<=1.0])

As discussed in the lecture, a lift of <1 could be seen as a substitution due to a negative correlation. In this case, it is (hopefully) not...

# Principal Component Analysis (PCA)

Last time we found out that the iris dataset was not easily plotable, because of its 4 dimensions. Now, let us apply a PCA to the iris dataset in order to visualize the data in 3D or even in 2D. PCA is available in `scikit-learn` or can be computed with `numpy`manually.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
import plotly as py
import plotly.graph_objs as go
from sklearn.preprocessing import StandardScaler

We load the data first:

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data , columns=iris.feature_names)
print(list(iris.target_names)) # print classes
ground_truth = iris.target
df.head()

## Normalize (z-score)

First of all, we need to normalize the data. Since PCA tries to find the PCs according to variance, the z-score normalization (= Standardization) is ideal:

In [ ]:
df_norm = StandardScaler().fit_transform(df)
df_norm = pd.DataFrame(df_norm, columns=['sepal length', 'sepal width', 'petal length', 'petal width' ]) # not in (cm) any more
df_norm.head()

## PCA with numpy

Now, let's compute the PCA by computing the covariance matrix and then the eigenvalues with `numpy`:

In [ ]:
cov_mat = np.cov(df_norm.T)
print(cov_mat)

Since we normalized the data to mean of 0 and variance of 1, the diagonal should contain values of (about) 1, since this is the variance of each feature. 

Now, let's compute the eigenvalues:

In [ ]:
eig_vals, eig_vecs = np.linalg.eig(cov_mat) # decomposition as discussed in the lecture

print('Eigenvalues', eig_vals)
print()
print('Eigenvectors\n', eig_vecs)

As you can see, the eigenvectors and eigenvalues show up as complex numbers, however the imaginary part is always zero. So let's remove it and keep only the relevant real part:

In [ ]:
eig_vecs = np.real(eig_vecs)
eig_vals = np.real(eig_vals)
print('Eigenvalues', eig_vals)
print()
print('Eigenvectors\n', eig_vecs)

Please note:
- The eigenvectors are in the columns of the eigenvector matrix. Fist column is first eigenvector (first PC).
- In total we have 4 eigenvectors
- The eigenvalues decrease: We already know that the first eigenvector has the highest eigenvalue (== variance in the "new" eigenspace)

If we want to project our dataset into the new coordinate system, we can simply calculate the dot product of the data and the eigenvectors:

In [ ]:
df_norm.dot(eig_vecs).head(10)

That was already the transformation. Now, if we want, we could remove the 4th (least important) dimension to perform a dimensionality reduction.

## PCA with scikit-learn

Since it is more comfortable, let's use the PCA functionality from `scikit-learn`from now on, which gives us the same results in one single method call. In the following, no parameter for PCA was used to get all principal components and eigenvalues. Typically when the number of dimensions should be reduced, the parameter `n_components`comes in handy. For example, if you only need two PCs for the iris data, set `n_components=2` in the following:

In [ ]:
pca = PCA() # typically you add here as a parameter the nbr. of cmponents: i.e.: n_components=2
            # we leave it blank to get all!
pcs = pca.fit_transform(df_norm) # principle components

Let's verify we got the same eigenvalues as above:

In [ ]:
pca.explained_variance_

Exactly the same result.

And check the principal components (the data being transformed into the new coordinate system consisting of the eigenvectors above:)

In [ ]:
print(pcs[:5])
print()
print(len(pcs))

Please note that in the following we are only using a subset of the columns. First, we create a 3D plot with the first three PCs, later a 2D plot with the 1st and 2nd PC only. The 3D plot is done with plotly:

In [ ]:
trace1 = go.Scatter3d(
    x= pcs[:,0],
    y= pcs[:,1],
    z= pcs[:,2],
    mode='markers',
     marker=dict(
        color = ground_truth, 
        size= 10,
        line=dict(
            color= ground_truth,
            width= 12
        ),
        opacity=0.8
     )
)
data = [trace1]
layout = go.Layout(
    height=700,
    title= 'Principal Components',
    scene = dict(
            xaxis = dict(title  = 'Principal Component 1'),
            yaxis = dict(title  = 'Principal Component 2'),
            zaxis = dict(title  = 'Principal Component 3')
        )
)
fig = go.Figure(data=data, layout=layout)
py.offline.iplot(fig)

Do you remeber the visualization from last week, where we always "left out" one dimension? This PCA is much better to grasp the distribution of the data! Please note the axis, these are not length/width of our flowers anymore, but a linear combination thereof. In short: They do not have any more meaning in terms of our original measurements.

Now, we plot the 1st and second PC only with matplotlib:

In [ ]:
fig = plt.figure()
ax = plt.axes()

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('PCA with 2 components')

targets = list(iris.target_names)
ax.scatter(pcs[ground_truth==0,0], 
            pcs[ground_truth==0,1],
            s=20,
            c='lightgreen', 
            marker='s', 
            label=targets[0])
ax.scatter(pcs[ground_truth==1,0], 
            pcs[ground_truth==1,1],
            s=20,
            c='orange', 
            marker='s', 
            label=targets[1])
ax.scatter(pcs[ground_truth==2,0], 
            pcs[ground_truth==2,1],
            s=20,
            c='blue', 
            marker='s', 
            label=targets[2])

ax.legend(targets)
ax.grid()
plt.show()

Interestingly, the 2D plot still seems fair, we can still see the separation of `setosa` from the other two well and also the mixture of the two other species look similar. So how many dimensions could we leave out without loosing too much information?

## Scree Plot and Kaiser Criteria

To find that out, lets have again a look at the eigenvalues:

In [ ]:
pca.explained_variance_

The eigenvalues tell us how much information (variance) can be attributed to each of the PCs. The higher the value, the more important. When we reduce the number of dimensions, we will loose the information of the last two PCs. It can also be interesting to look at the percentage of variance, each PC contains:

In [ ]:
pca.explained_variance_ratio_

We can see that the first principal component contains 73.0 percent of the variance, and the second principal component contains 22.9 percent of the variance. Together, the two components contain 95.9 percent of the "information"!

A rule of thumb, the so called **Kaiser criteria** tells us, that we can (potentially) remove a dimension, if the eigenvalue is less than 1.0. To check that, we plot them in the so called **Scree Plot**

In [ ]:
import matplotlib.ticker as ticker
fig = plt.figure()
ax = plt.axes()

pc_values = np.arange(pca.n_components_) + 1
ax.plot(pc_values, pca.explained_variance_, 'o-', linewidth=2, color='blue')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue')
plt.axhline(y=1, linewidth=1, color='r')
plt.show()

As can be seen from the plot, theoretically one dimension would be OK according to the rule of thumb. However, since the second PC is very close, 2D might be better in our case.

**Please note**: Please don't start to remove dimensions now as a default step. It is helpful for visualizing high dimensional data and in some cases also as a preprocessing step (c.f. curse of dimensionality). However, you will loose information and also the meaning of your axis!